# Analisi Strutturale della Rete Normativa

Questo notebook analizza la struttura della rete normativa del grafo focale e verifica se la gerarchia delle fonti (framework Lamfalussy) è riflessa nella topologia della rete di citazioni.

## Ipotesi di ricerca

Se la struttura normativa è razionale e coerente con il framework Lamfalussy, ci aspettiamo:

1. **L1 > L2 > L3 per indegree medio** — gli atti legislativi quadro vengono citati più degli atti di attuazione, che vengono citati più della soft law
2. **L1 > L2 per betweenness** — gli atti L1 sono i nodi ponte della rete, connettono sottografi tematici diversi
3. **L4 outdegree ≈ 0** — le sentenze non citano atti normativi (il flusso è unidirezionale verso la giurisprudenza)
4. **Correlazione negativa indegree/outdegree per layer** — i nodi citati (L1) citano poco, i nodi che citano molto (L2/L3) vengono citati poco

## Input/Output
- **Input**: file esportato da Gephi con metriche calcolate
- **Output**: `data/output/{materia}/network_analysis_report.csv` + grafici

> **Replicabilità**: questo notebook non contiene parametri specifici per materia. Cambiare `INPUT_FILE` per analizzare un corpus diverso.

## 0. Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import os
import sys

sys.path.append('..')
from config_golden_power import MATERIA_NAME

output_path = os.path.join('..', 'data', 'output', MATERIA_NAME)
INPUT_FILE  = os.path.join(output_path, 'gephi_nodes_exported.csv')
OUTPUT_CSV  = os.path.join(output_path, 'network_analysis_report.csv')
OUTPUT_FIGS = output_path  # salva i grafici nella stessa cartella

# Palette layer — coerente con i colori Gephi
LAYER_COLORS = {'L1': '#8B5CF6', 'L2': '#10B981', 'L3': '#F59E0B', 'L4': '#EF4444'}
LAYER_ORDER  = ['L1', 'L2', 'L3', 'L4']

df = pd.read_csv(INPUT_FILE)

print(f"Nodi caricati:  {len(df)}")
print(f"Colonne:        {df.columns.tolist()}")
print()
print("Distribuzione layer:")
print(df['layer'].value_counts().to_string())

## 1. Statistiche Descrittive per Layer

Per ciascun layer calcoliamo le metriche di rete principali: indegree, outdegree, degree totale, betweenness centrality, closeness centrality, clustering coefficient.

In [ ]:
METRICS = ['indegree', 'outdegree', 'degree', 'betweenesscentrality',
           'closnesscentrality', 'clustering']

# Statistiche per layer
stats = df.groupby('layer')[METRICS].agg(['mean', 'median', 'std', 'max'])
stats.columns = ['_'.join(c) for c in stats.columns]
stats = stats.loc[[l for l in LAYER_ORDER if l in stats.index]]

print("=== STATISTICHE PER LAYER ===")
print()

# Stampa leggibile per le metriche principali
for metric in ['indegree', 'outdegree', 'betweenesscentrality']:
    print(f"{metric}:")
    for layer in LAYER_ORDER:
        if layer not in df['layer'].values:
            continue
        m   = stats.loc[layer, f'{metric}_mean']
        med = stats.loc[layer, f'{metric}_median']
        mx  = stats.loc[layer, f'{metric}_max']
        n   = (df['layer'] == layer).sum()
        print(f"  {layer} (n={n:>4})  media={m:>8.2f}  mediana={med:>8.2f}  max={mx:>8.2f}")
    print()

## 2. Verifica delle Ipotesi

Verifica formale delle ipotesi enunciate nell'introduzione.

In [ ]:
def check_hypothesis(label, condition, value, expected):
    """Stampa il risultato di una verifica con valore osservato e atteso."""
    status = 'CONFERMATA' if condition else 'NON CONFERMATA'
    print(f"  [{status}] {label}")
    print(f"             Osservato: {value}  |  Atteso: {expected}")
    print()

means = df.groupby('layer')['indegree'].mean()
means_out = df.groupby('layer')['outdegree'].mean()
means_btw = df.groupby('layer')['betweenesscentrality'].mean()

print("=== VERIFICA IPOTESI ===")
print()

# H1: L1 indegree > L2 indegree
check_hypothesis(
    "H1: indegree L1 > L2 (L1 più citato)",
    means.get('L1', 0) > means.get('L2', 0),
    f"L1={means.get('L1',0):.2f}, L2={means.get('L2',0):.2f}",
    "L1 > L2"
)

# H2: L1 betweenness > L2 betweenness
check_hypothesis(
    "H2: betweenness L1 > L2 (L1 sono nodi ponte)",
    means_btw.get('L1', 0) > means_btw.get('L2', 0),
    f"L1={means_btw.get('L1',0):.1f}, L2={means_btw.get('L2',0):.1f}",
    "L1 > L2"
)

# H3: L4 outdegree ≈ 0
l4_outdegree = means_out.get('L4', 0)
check_hypothesis(
    "H3: outdegree L4 ≈ 0 (sentenze non citano atti)",
    l4_outdegree < 0.5,
    f"L4={l4_outdegree:.2f}",
    "≈ 0"
)

# H4: correlazione negativa indegree/outdegree per L1
corr_l1 = df[df['layer'] == 'L1'][['indegree', 'outdegree']].corr().iloc[0, 1]
check_hypothesis(
    "H4: correlazione negativa indegree/outdegree in L1",
    corr_l1 < 0,
    f"r={corr_l1:.3f}",
    "r < 0"
)

## 3. Distribuzione Indegree e Outdegree per Layer

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Distribuzione Indegree e Outdegree per Layer', fontsize=13, fontweight='bold')

for ax, metric, label in zip(axes, ['indegree', 'outdegree'], ['Indegree', 'Outdegree']):
    data_by_layer = [
        df[df['layer'] == l][metric].values
        for l in LAYER_ORDER if l in df['layer'].values
    ]
    labels_present = [l for l in LAYER_ORDER if l in df['layer'].values]
    colors_present = [LAYER_COLORS[l] for l in labels_present]

    bp = ax.boxplot(data_by_layer, labels=labels_present, patch_artist=True,
                    medianprops=dict(color='black', linewidth=2),
                    showfliers=False)  # nascondi outlier estremi per leggibilità

    for patch, color in zip(bp['boxes'], colors_present):
        patch.set_facecolor(color)
        patch.set_alpha(0.7)

    # Aggiungi media come punto
    for i, (layer, d) in enumerate(zip(labels_present, data_by_layer)):
        ax.scatter(i + 1, np.mean(d), color='black', zorder=5, s=40, marker='D')

    ax.set_title(label, fontsize=11)
    ax.set_xlabel('Layer')
    ax.set_ylabel('Numero di archi')
    ax.yaxis.set_major_locator(mticker.MaxNLocator(integer=True))
    ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_FIGS, 'fig_degree_distribution.png'), dpi=150, bbox_inches='tight')
plt.show()
print("Salvato: fig_degree_distribution.png")

## 4. Betweenness Centrality per Layer

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

layer_means = df.groupby('layer')['betweenesscentrality'].mean().reindex(LAYER_ORDER).dropna()
layer_se    = df.groupby('layer')['betweenesscentrality'].sem().reindex(LAYER_ORDER).dropna()

bars = ax.bar(
    layer_means.index,
    layer_means.values,
    yerr=layer_se.values,
    color=[LAYER_COLORS[l] for l in layer_means.index],
    alpha=0.8,
    capsize=4,
    edgecolor='white'
)

ax.set_title('Betweenness Centrality Media per Layer', fontsize=12, fontweight='bold')
ax.set_xlabel('Layer')
ax.set_ylabel('Betweenness centrality (media ± SE)')
ax.grid(axis='y', alpha=0.3)

for bar, val in zip(bars, layer_means.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
            f'{val:.0f}', ha='center', va='bottom', fontsize=10)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_FIGS, 'fig_betweenness_by_layer.png'), dpi=150, bbox_inches='tight')
plt.show()
print("Salvato: fig_betweenness_by_layer.png")

## 5. Top Nodi per Indegree

I nodi con più indegree sono quelli più citati nella rete — i riferimenti normativi fondamentali della materia.

In [ ]:
N_TOP = 15

top_nodes = df.nlargest(N_TOP, 'indegree')[[
    'Label', 'layer', 'layer_confidence', 'indegree', 'outdegree',
    'betweenesscentrality', 'year', 'title'
]].copy()

top_nodes['title_short'] = top_nodes['title'].apply(
    lambda t: str(t)[:90] + '...' if pd.notna(t) and len(str(t)) > 90 else str(t)
)

print(f"TOP {N_TOP} NODI PER INDEGREE")
print()
for _, r in top_nodes.iterrows():
    print(f"  {r['layer']} ({r['layer_confidence']:<6}) | "
          f"in={r['indegree']:>3}  out={r['outdegree']:>3}  "
          f"btw={r['betweenesscentrality']:>8.0f} | "
          f"{r['title_short']}")

# Grafico
fig, ax = plt.subplots(figsize=(12, 6))
colors  = [LAYER_COLORS.get(l, '#999') for l in top_nodes['layer']]
y_pos   = range(len(top_nodes))

ax.barh(list(y_pos), top_nodes['indegree'].values, color=colors, alpha=0.8, edgecolor='white')
ax.set_yticks(list(y_pos))
ax.set_yticklabels(top_nodes['Label'].values, fontsize=9)
ax.set_xlabel('Indegree')
ax.set_title(f'Top {N_TOP} nodi per indegree', fontsize=12, fontweight='bold')
ax.invert_yaxis()
ax.grid(axis='x', alpha=0.3)

# Legenda layer
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor=LAYER_COLORS[l], label=l) for l in LAYER_ORDER]
ax.legend(handles=legend_elements, loc='lower right')

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_FIGS, 'fig_top_nodes_indegree.png'), dpi=150, bbox_inches='tight')
plt.show()
print("Salvato: fig_top_nodes_indegree.png")

## 6. Ratio Indegree/Outdegree per Layer

Un ratio > 1 indica che il nodo riceve più citazioni di quante ne produce — caratteristica attesa per L1. Un ratio < 1 indica il contrario — caratteristica attesa per L2/L3.

In [ ]:
# Calcola ratio (evita divisione per zero)
df['in_out_ratio'] = df.apply(
    lambda r: r['indegree'] / r['outdegree'] if r['outdegree'] > 0
         else (float('inf') if r['indegree'] > 0 else None),
    axis=1
)

print("Ratio indegree/outdegree per layer (mediana, esclusi nodi con outdegree=0):")
ratio_stats = df[df['outdegree'] > 0].groupby('layer')['in_out_ratio'].agg(['median', 'mean'])
print(ratio_stats.reindex(LAYER_ORDER).dropna().round(2).to_string())
print()

# Quota di nodi con indegree > outdegree per layer
print("% nodi con indegree > outdegree per layer:")
for layer in LAYER_ORDER:
    sub = df[df['layer'] == layer]
    pct = (sub['indegree'] > sub['outdegree']).sum() / len(sub) * 100
    print(f"  {layer}: {pct:.1f}%  (n={len(sub)})")

## 7. Evoluzione Temporale per Layer

Quanti atti per layer sono stati prodotti in ciascuna decade — mostra come la struttura normativa della materia si è evoluta nel tempo.

In [ ]:
temporal = df[df['year'].notna() & df['layer'].notna()].copy()
temporal['decade'] = (temporal['year'] // 10 * 10).astype(int)

pivot = temporal.groupby(['decade', 'layer']).size().unstack(fill_value=0)
pivot = pivot[[l for l in LAYER_ORDER if l in pivot.columns]]
pivot = pivot[pivot.index >= 1990]  # dal 1990 in poi per leggibilità

fig, ax = plt.subplots(figsize=(12, 5))

bottom = np.zeros(len(pivot))
for layer in pivot.columns:
    values = pivot[layer].values
    ax.bar(pivot.index, values, bottom=bottom, label=layer,
           color=LAYER_COLORS[layer], alpha=0.85, width=8, edgecolor='white')
    bottom += values

ax.set_title('Produzione Normativa per Decade e Layer (dal 1990)', fontsize=12, fontweight='bold')
ax.set_xlabel('Decade')
ax.set_ylabel('Numero di atti')
ax.set_xticks(pivot.index)
ax.set_xticklabels([f"{d}s" for d in pivot.index])
ax.legend(title='Layer', loc='upper left')
ax.grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_FIGS, 'fig_temporal_by_layer.png'), dpi=150, bbox_inches='tight')
plt.show()
print("Salvato: fig_temporal_by_layer.png")

## 8. Export Report

In [ ]:
# Tabella riassuntiva per layer
report = df.groupby('layer').agg(
    n_nodes                = ('Id', 'count'),
    indegree_mean          = ('indegree', 'mean'),
    indegree_median        = ('indegree', 'median'),
    indegree_max           = ('indegree', 'max'),
    outdegree_mean         = ('outdegree', 'mean'),
    outdegree_median       = ('outdegree', 'median'),
    betweenness_mean       = ('betweenesscentrality', 'mean'),
    betweenness_max        = ('betweenesscentrality', 'max'),
    clustering_mean        = ('clustering', 'mean'),
    pct_high_confidence    = ('layer_confidence', lambda x: (x == 'high').sum() / len(x) * 100),
).reindex(LAYER_ORDER).round(2)

report.to_csv(OUTPUT_CSV)

print(f"Report salvato: {OUTPUT_CSV}")
print()
print("=" * 60)
print("RIEPILOGO ANALISI STRUTTURALE")
print("=" * 60)
print(report.to_string())
print()
print("Figure salvate:")
for fig_name in ['fig_degree_distribution.png', 'fig_betweenness_by_layer.png',
                  'fig_top_nodes_indegree.png', 'fig_temporal_by_layer.png']:
    path = os.path.join(OUTPUT_FIGS, fig_name)
    exists = os.path.exists(path)
    print(f"  {'OK' if exists else 'MANCANTE'} {fig_name}")